In [1]:
!pip install transformers datasets accelerate scikit-learn sentencepiece protobuf -q

In [2]:
from google.colab import drive
import os, zipfile

drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/deberta-v3-small.zip"
cache_dir = os.path.expanduser("~/.cache/huggingface/hub/")
os.makedirs(cache_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    for info in z.infolist():
        fixed_name = info.filename.replace("\\", "/")
        out_path = os.path.join(cache_dir, fixed_name)
        if info.is_dir() or fixed_name.endswith("/"):
            os.makedirs(out_path, exist_ok=True)
        elif info.file_size == 0:
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
        else:
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
            with z.open(info) as src, open(out_path, 'wb') as dst:
                dst.write(src.read())

model_cache = os.path.join(cache_dir, "models--microsoft--deberta-v3-small")
print(f"Model extracted to: {model_cache}")
print(f"Contents: {os.listdir(model_cache)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model extracted to: /root/.cache/huggingface/hub/models--microsoft--deberta-v3-small
Contents: ['snapshots', 'blobs', 'refs']


In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
import json
import os
import gc
import re
import nltk
from collections import defaultdict
import random

os.environ["HF_HUB_OFFLINE"] = "1"

from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [4]:
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [w for w in text.split() if w not in stop_words]
    return ' '.join(tokens)

df_raw = pd.read_csv('ExioNAICS.csv')

naics6 = df_raw[['NAICS Code', 'NAICS Title', 'Description']].drop_duplicates(subset='NAICS Code').copy()
naics6['NAICS Code'] = naics6['NAICS Code'].astype(str)
naics6['clean_text'] = (naics6['NAICS Title'] + ' ' + naics6['Description'].fillna('')).apply(preprocess_text)
naics6 = naics6.reset_index(drop=True)

naics3 = df_raw[['NAICS_3 Code', 'NAICS_3 Title', 'NAICS_3 Description']].drop_duplicates(subset='NAICS_3 Code').copy()
naics3.columns = ['code', 'title', 'description']
naics3['code'] = naics3['code'].astype(str)
naics3['clean_text'] = (naics3['title'] + ' ' + naics3['description'].fillna('')).apply(preprocess_text)
naics3 = naics3.reset_index(drop=True)

naics2 = df_raw[['NAICS_2 Code', 'NAICS_2 Title', 'NAICS_2 Description']].drop_duplicates(subset='NAICS_2 Code').copy()
naics2.columns = ['code', 'title', 'description']
naics2['code'] = naics2['code'].astype(str)
naics2['clean_text'] = (naics2['title'] + ' ' + naics2['description'].fillna('')).apply(preprocess_text)
naics2 = naics2.reset_index(drop=True)

code6_to_idx = {code: i for i, code in enumerate(naics6['NAICS Code'])}
idx_to_code6 = {i: code for code, i in code6_to_idx.items()}

code6_to_code3 = {}
code6_to_code2 = {}
for _, row in df_raw[['NAICS Code', 'NAICS_3 Code', 'NAICS_2 Code']].drop_duplicates().iterrows():
    c6 = str(row['NAICS Code'])
    code6_to_code3[c6] = str(row['NAICS_3 Code'])
    code6_to_code2[c6] = str(row['NAICS_2 Code'])

code3_to_code6_indices = defaultdict(list)
code2_to_code3_codes = defaultdict(set)
for c6, idx in code6_to_idx.items():
    c3 = code6_to_code3.get(c6)
    c2 = code6_to_code2.get(c6)
    if c3:
        code3_to_code6_indices[c3].append(idx)
    if c2 and c3:
        code2_to_code3_codes[c2].add(c3)

corpus_texts_6 = naics6['clean_text'].tolist()
corpus_texts_3 = naics3['clean_text'].tolist()
corpus_texts_2 = naics2['clean_text'].tolist()

code3_to_idx3 = {code: i for i, code in enumerate(naics3['code'])}
code2_to_idx2 = {code: i for i, code in enumerate(naics2['code'])}

print(f"NAICS hierarchy:")
print(f"  Level 2: {len(naics2)} sectors")
print(f"  Level 3: {len(naics3)} subsectors")
print(f"  Level 6: {len(naics6)} industries")

df = pd.read_csv('ExioNAICS_preprocessed.csv')
df['NAICS Code'] = df['NAICS Code'].astype(str)
df['naics_idx'] = df['NAICS Code'].map(code6_to_idx)
df['naics2'] = df['NAICS Code'].map(code6_to_code2)

missing = df['naics_idx'].isna().sum()
if missing > 0:
    df = df.dropna(subset=['naics_idx']).reset_index(drop=True)
df['naics_idx'] = df['naics_idx'].astype(int)

print(f"\nDataset: {len(df)} samples, {df['naics_idx'].nunique()} unique codes")

NAICS hierarchy:
  Level 2: 24 sectors
  Level 3: 94 subsectors
  Level 6: 1115 industries

Dataset: 20535 samples, 1115 unique codes


In [5]:
MODEL_NAME = "microsoft/deberta-v3-small"
MAX_LENGTH = 128
BATCH_SIZE = 128
NUM_EPOCHS = 50
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
TEMPERATURE = 0.05
PROJECTION_DIM = 256
SEED = 42
VAL_RATIO = 0.1
PATIENCE = 10
GROUP_REASONING_K2 = 5
GROUP_REASONING_K3 = 15

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(f"=== Enhanced Contrastive Learning + Hierarchical Group Reasoning ===")
print(f"  Model:           {MODEL_NAME}")
print(f"  Projection dim:  {PROJECTION_DIM}")
print(f"  Batch size:      {BATCH_SIZE} (3x more negatives)")
print(f"  Max length:      {MAX_LENGTH}")
print(f"  Epochs:          {NUM_EPOCHS}")
print(f"  Learning rate:   {LEARNING_RATE}")
print(f"  Temperature:     {TEMPERATURE}")
print(f"  Patience:        {PATIENCE}")
print(f"  Group Reasoning: k2={GROUP_REASONING_K2}, k3={GROUP_REASONING_K3}")

=== Enhanced Contrastive Learning + Hierarchical Group Reasoning ===
  Model:           microsoft/deberta-v3-small
  Projection dim:  256
  Batch size:      128 (3x more negatives)
  Max length:      128
  Epochs:          50
  Learning rate:   2e-05
  Temperature:     0.05
  Patience:        10
  Group Reasoning: k2=5, k3=15


In [6]:
class DeBERTaEncoder(nn.Module):
    def __init__(self, model_name, projection_dim=256):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name, torch_dtype=torch.float32)
        hidden = self.backbone.config.hidden_size
        self.projection = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.LayerNorm(hidden),
            nn.Linear(hidden, projection_dim),
        )
        self.hidden_size = projection_dim

    def mean_pool(self, token_embeddings, attention_mask):
        mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        return sum_embeddings / sum_mask

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.mean_pool(outputs.last_hidden_state, attention_mask)
        projected = self.projection(pooled)
        return F.normalize(projected, p=2, dim=1)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = DeBERTaEncoder(MODEL_NAME, PROJECTION_DIM).to(device)

total_params = sum(p.numel() for p in model.parameters())
proj_params = sum(p.numel() for p in model.projection.parameters())
print(f"Total params: {total_params:,}")
print(f"Projection head params: {proj_params:,}")
print(f"Output dim: {PROJECTION_DIM}")

test_input = tokenizer("test sentence", return_tensors="pt", max_length=MAX_LENGTH, truncation=True, padding=True)
test_input = {k: v.to(device) for k, v in test_input.items() if k in ['input_ids', 'attention_mask']}
with torch.no_grad():
    test_emb = model(**test_input)
print(f"Output shape: {test_emb.shape}, norm: {test_emb.norm().item():.4f}, NaN: {torch.isnan(test_emb).any().item()}")

The tokenizer you are loading from 'microsoft/deberta-v3-small' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total params: 142,093,312
Projection head params: 788,992
Output dim: 256
Output shape: torch.Size([1, 256]), norm: 1.0000, NaN: False


In [7]:
def mnr_loss(query_embeddings, doc_embeddings, temperature=0.05):
    similarity = torch.matmul(query_embeddings, doc_embeddings.T) / temperature
    labels = torch.arange(similarity.size(0), device=similarity.device)
    loss_qd = F.cross_entropy(similarity, labels)
    loss_dq = F.cross_entropy(similarity.T, labels)
    return (loss_qd + loss_dq) / 2

print("Symmetric MNR loss defined (query→doc + doc→query)")

Symmetric MNR loss defined (query→doc + doc→query)


In [8]:
class ContrastiveDataset(Dataset):
    def __init__(self, descriptions, naics_indices, corpus_texts, tokenizer, max_length):
        self.descriptions = descriptions
        self.naics_indices = naics_indices
        self.corpus_texts = corpus_texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.descriptions)

    def __getitem__(self, idx):
        query = self.descriptions[idx]
        doc = self.corpus_texts[self.naics_indices[idx]]

        query_enc = self.tokenizer(
            query, max_length=self.max_length, truncation=True,
            padding='max_length', return_tensors='pt'
        )
        doc_enc = self.tokenizer(
            doc, max_length=self.max_length, truncation=True,
            padding='max_length', return_tensors='pt'
        )

        return {
            'query_input_ids': query_enc['input_ids'].squeeze(0),
            'query_attention_mask': query_enc['attention_mask'].squeeze(0),
            'doc_input_ids': doc_enc['input_ids'].squeeze(0),
            'doc_attention_mask': doc_enc['attention_mask'].squeeze(0),
            'naics_idx': self.naics_indices[idx],
        }


class HardNegativeSampler(Sampler):
    def __init__(self, naics2_labels, batch_size, num_groups=16):
        self.batch_size = batch_size
        self.num_groups = num_groups
        self.group_to_indices = defaultdict(list)
        for i, label in enumerate(naics2_labels):
            self.group_to_indices[label].append(i)
        self.groups = list(self.group_to_indices.keys())
        self.n = len(naics2_labels)

    def __iter__(self):
        all_indices = []
        per_group = self.batch_size // self.num_groups
        remainder = self.batch_size % self.num_groups

        num_batches = self.n // self.batch_size
        for _ in range(num_batches):
            batch = []
            selected_groups = random.choices(self.groups, k=self.num_groups)
            for gi, g in enumerate(selected_groups):
                count = per_group + (1 if gi < remainder else 0)
                pool = self.group_to_indices[g]
                batch.extend(random.choices(pool, k=count))
            random.shuffle(batch)
            all_indices.extend(batch)

        leftover = list(range(self.n))
        random.shuffle(leftover)
        all_indices.extend(leftover[:self.n - len(all_indices)])
        return iter(all_indices)

    def __len__(self):
        return self.n

print("ContrastiveDataset + HardNegativeSampler defined")

ContrastiveDataset + HardNegativeSampler defined


In [9]:
label_counts = df['naics_idx'].value_counts()
valid_labels = label_counts[label_counts >= 2].index
df = df[df['naics_idx'].isin(valid_labels)].reset_index(drop=True)

train_idx, val_idx = train_test_split(
    np.arange(len(df)), test_size=VAL_RATIO, random_state=SEED, stratify=df['naics_idx']
)

train_descriptions = df['clean_description'].iloc[train_idx].tolist()
train_naics_indices = df['naics_idx'].iloc[train_idx].tolist()
train_naics2 = df['naics2'].iloc[train_idx].tolist()
val_descriptions = df['clean_description'].iloc[val_idx].tolist()
val_naics_indices = df['naics_idx'].iloc[val_idx].tolist()

train_dataset = ContrastiveDataset(train_descriptions, train_naics_indices, corpus_texts_6, tokenizer, MAX_LENGTH)
val_dataset = ContrastiveDataset(val_descriptions, val_naics_indices, corpus_texts_6, tokenizer, MAX_LENGTH)

hard_sampler = HardNegativeSampler(train_naics2, BATCH_SIZE, num_groups=16)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=hard_sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"After filtering: {len(df)} samples, {df['naics_idx'].nunique()} codes")
print(f"Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_dataset)} samples, {len(val_loader)} batches")
print(f"Hard negative sampling: {hard_sampler.num_groups} NAICS-2 groups per batch")

After filtering: 20533 samples, 1113 codes
Train: 18479 samples, 145 batches
Val:   2054 samples, 17 batches
Hard negative sampling: 16 NAICS-2 groups per batch


In [10]:
@torch.no_grad()
def encode_texts(model, texts, tokenizer, max_length, batch_size=128):
    model.eval()
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        enc = tokenizer(
            batch_texts, max_length=max_length, truncation=True,
            padding=True, return_tensors='pt'
        )
        enc = {k: v.to(device) for k, v in enc.items() if k in ['input_ids', 'attention_mask']}
        emb = model(**enc)
        all_embeddings.append(emb.cpu())
    return torch.cat(all_embeddings, dim=0)


@torch.no_grad()
def evaluate_flat(model, val_loader, corpus_emb_6):
    model.eval()
    all_query_embs = []
    all_labels = []
    for batch in val_loader:
        q = model(batch['query_input_ids'].to(device), batch['query_attention_mask'].to(device))
        all_query_embs.append(q.cpu())
        all_labels.append(batch['naics_idx'])
    query_embs = torch.cat(all_query_embs, dim=0)
    labels = torch.cat(all_labels, dim=0)
    sim = torch.matmul(query_embs, corpus_emb_6.T)
    n = len(labels)
    top1, top5, top10 = 0, 0, 0
    for i in range(n):
        topk = sim[i].topk(10).indices
        t = labels[i].item()
        if t == topk[0].item(): top1 += 1
        if t in topk[:5]: top5 += 1
        if t in topk[:10]: top10 += 1
    return {'top1': top1/n, 'top5': top5/n, 'top10': top10/n}


@torch.no_grad()
def evaluate_hierarchical(model, val_loader, corpus_emb_2, corpus_emb_3, corpus_emb_6,
                          code2_to_code3_codes, code3_to_code6_indices,
                          naics2, naics3, code3_to_idx3, k2=5, k3=15):
    model.eval()
    all_query_embs = []
    all_labels = []
    for batch in val_loader:
        q = model(batch['query_input_ids'].to(device), batch['query_attention_mask'].to(device))
        all_query_embs.append(q.cpu())
        all_labels.append(batch['naics_idx'])
    query_embs = torch.cat(all_query_embs, dim=0)
    labels = torch.cat(all_labels, dim=0)
    n = len(labels)

    sim2 = torch.matmul(query_embs, corpus_emb_2.T)

    top1, top5, top10 = 0, 0, 0
    for i in range(n):
        top_k2_indices = sim2[i].topk(k2).indices.tolist()
        top_k2_codes = [naics2['code'].iloc[j] for j in top_k2_indices]

        candidate_c3_idx = []
        for c2 in top_k2_codes:
            for c3 in code2_to_code3_codes.get(c2, []):
                if c3 in code3_to_idx3:
                    candidate_c3_idx.append(code3_to_idx3[c3])
        if not candidate_c3_idx:
            continue

        candidate_c3_idx = list(set(candidate_c3_idx))
        c3_embs = corpus_emb_3[candidate_c3_idx]
        sim3 = torch.matmul(query_embs[i].unsqueeze(0), c3_embs.T).squeeze(0)
        top_k3_local = sim3.topk(min(k3, len(candidate_c3_idx))).indices.tolist()
        top_k3_codes = [naics3['code'].iloc[candidate_c3_idx[j]] for j in top_k3_local]

        candidate_c6_idx = []
        for c3 in top_k3_codes:
            candidate_c6_idx.extend(code3_to_code6_indices.get(c3, []))
        if not candidate_c6_idx:
            continue

        candidate_c6_idx = list(set(candidate_c6_idx))
        c6_embs = corpus_emb_6[candidate_c6_idx]
        sim6 = torch.matmul(query_embs[i].unsqueeze(0), c6_embs.T).squeeze(0)
        top_k6_local = sim6.topk(min(10, len(candidate_c6_idx))).indices.tolist()
        top_k6_global = [candidate_c6_idx[j] for j in top_k6_local]

        t = labels[i].item()
        if top_k6_global[0] == t: top1 += 1
        if t in top_k6_global[:5]: top5 += 1
        if t in top_k6_global[:10]: top10 += 1

    return {'top1': top1/n, 'top5': top5/n, 'top10': top10/n}

print("Evaluation functions defined: flat MIPS + hierarchical group reasoning")

Evaluation functions defined: flat MIPS + hierarchical group reasoning


In [11]:
from transformers import get_cosine_schedule_with_warmup

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

os.makedirs("results", exist_ok=True)
best_top1 = 0.0
best_epoch = 0
patience_counter = 0
epoch_log = []

print(f"Optimizer: AdamW, lr={LEARNING_RATE}, wd={WEIGHT_DECAY}")
print(f"Scheduler: cosine with warmup ({warmup_steps} steps)")
print(f"Total steps: {total_steps}")
print(f"\n{'='*95}")
print(f"{'Ep':>3} {'TrLoss':>8} {'Flat1':>7} {'Flat5':>7} {'Flat10':>7} {'Hier1':>7} {'Hier5':>7} {'Hier10':>7} {'LR':>10}")
print(f"{'='*95}")

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    num_batches = 0

    for batch in train_loader:
        query_emb = model(batch['query_input_ids'].to(device), batch['query_attention_mask'].to(device))
        doc_emb = model(batch['doc_input_ids'].to(device), batch['doc_attention_mask'].to(device))
        loss = mnr_loss(query_emb, doc_emb, temperature=TEMPERATURE)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        num_batches += 1

    avg_loss = total_loss / num_batches
    lr = scheduler.get_last_lr()[0]

    corpus_emb_6 = encode_texts(model, corpus_texts_6, tokenizer, MAX_LENGTH)
    corpus_emb_3 = encode_texts(model, corpus_texts_3, tokenizer, MAX_LENGTH)
    corpus_emb_2 = encode_texts(model, corpus_texts_2, tokenizer, MAX_LENGTH)

    flat = evaluate_flat(model, val_loader, corpus_emb_6)
    hier = evaluate_hierarchical(
        model, val_loader, corpus_emb_2, corpus_emb_3, corpus_emb_6,
        code2_to_code3_codes, code3_to_code6_indices,
        naics2, naics3, code3_to_idx3,
        k2=GROUP_REASONING_K2, k3=GROUP_REASONING_K3
    )

    best_this_epoch = max(flat['top1'], hier['top1'])

    epoch_log.append({
        'epoch': epoch, 'train_loss': avg_loss, 'lr': lr,
        'flat_top1': flat['top1'], 'flat_top5': flat['top5'], 'flat_top10': flat['top10'],
        'hier_top1': hier['top1'], 'hier_top5': hier['top5'], 'hier_top10': hier['top10'],
    })

    print(f"{epoch:>3} {avg_loss:>8.4f} {flat['top1']:>7.4f} {flat['top5']:>7.4f} {flat['top10']:>7.4f} "
          f"{hier['top1']:>7.4f} {hier['top5']:>7.4f} {hier['top10']:>7.4f} {lr:>10.2e}")

    if best_this_epoch > best_top1:
        best_top1 = best_this_epoch
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), "results/best_model.pt")
    else:
        patience_counter += 1

    pd.DataFrame(epoch_log).to_csv("results/hierarchical_epoch_log.csv", index=False)

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

print(f"\n{'='*95}")
print(f"Best Top-1: {best_top1:.4f} at epoch {best_epoch}")
model.load_state_dict(torch.load("results/best_model.pt", weights_only=True))
print("Loaded best model.")

Optimizer: AdamW, lr=2e-05, wd=0.01
Scheduler: cosine with warmup (725 steps)
Total steps: 7250

 Ep   TrLoss   Flat1   Flat5  Flat10   Hier1   Hier5  Hier10         LR
  1   4.8909  0.0068  0.0204  0.0341  0.0044  0.0195  0.0365   4.00e-06
  2   4.1434  0.0326  0.1013  0.1597  0.0292  0.0925  0.1519   8.00e-06
  3   3.3552  0.0701  0.2181  0.3325  0.0623  0.1996  0.2887   1.20e-05
  4   2.8621  0.1022  0.2809  0.4007  0.0910  0.2347  0.3306   1.60e-05
  5   2.5495  0.1217  0.3315  0.4348  0.1052  0.2634  0.3554   2.00e-05
  6   2.2863  0.1334  0.3471  0.4645  0.1139  0.2911  0.3778   2.00e-05
  7   2.0977  0.1388  0.3754  0.4966  0.1198  0.3228  0.4275   1.99e-05
  8   1.9463  0.1495  0.3729  0.4951  0.1207  0.2858  0.3700   1.98e-05
  9   1.8002  0.1500  0.3832  0.4942  0.1334  0.3233  0.4182   1.96e-05
 10   1.7185  0.1465  0.3875  0.5097  0.1344  0.3296  0.4284   1.94e-05
 11   1.6430  0.1465  0.3832  0.5000  0.1232  0.3062  0.3856   1.91e-05
 12   1.5068  0.1431  0.3749  0.4981  0

In [12]:
corpus_emb_6 = encode_texts(model, corpus_texts_6, tokenizer, MAX_LENGTH)
corpus_emb_3 = encode_texts(model, corpus_texts_3, tokenizer, MAX_LENGTH)
corpus_emb_2 = encode_texts(model, corpus_texts_2, tokenizer, MAX_LENGTH)

flat = evaluate_flat(model, val_loader, corpus_emb_6)
hier = evaluate_hierarchical(
    model, val_loader, corpus_emb_2, corpus_emb_3, corpus_emb_6,
    code2_to_code3_codes, code3_to_code6_indices,
    naics2, naics3, code3_to_idx3,
    k2=GROUP_REASONING_K2, k3=GROUP_REASONING_K3
)

print("=== Final Results (Best Model) ===")
print(f"\n  Flat Retrieval:")
print(f"    Top-1:  {flat['top1']:.4f}")
print(f"    Top-5:  {flat['top5']:.4f}")
print(f"    Top-10: {flat['top10']:.4f}")
print(f"\n  Hierarchical Group Reasoning (k2={GROUP_REASONING_K2}, k3={GROUP_REASONING_K3}):")
print(f"    Top-1:  {hier['top1']:.4f}")
print(f"    Top-5:  {hier['top5']:.4f}")
print(f"    Top-10: {hier['top10']:.4f}")
print(f"\n  Best epoch: {best_epoch}")

results = {
    "method": "DeBERTa-v3-small + MNR + Projection Head + Hard Negatives + Group Reasoning",
    "config": {
        "model": MODEL_NAME, "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE, "temperature": TEMPERATURE,
        "projection_dim": PROJECTION_DIM, "weight_decay": WEIGHT_DECAY,
        "num_epochs_trained": best_epoch,
        "group_reasoning_k2": GROUP_REASONING_K2, "group_reasoning_k3": GROUP_REASONING_K3,
    },
    "flat_results": flat,
    "hierarchical_results": hier,
}

with open("results/hierarchical_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

import zipfile, glob
with zipfile.ZipFile("results.zip", "w") as zf:
    for f in glob.glob("results/*.json") + glob.glob("results/*.csv") + glob.glob("results/*.pt"):
        zf.write(f)
        print(f"  Added: {f}")

from google.colab import files
files.download("results.zip")
print("\nDownloaded results.zip")

=== Final Results (Best Model) ===

  Flat Retrieval:
    Top-1:  0.1500
    Top-5:  0.3832
    Top-10: 0.4942

  Hierarchical Group Reasoning (k2=5, k3=15):
    Top-1:  0.1334
    Top-5:  0.3233
    Top-10: 0.4182

  Best epoch: 9
  Added: results/hierarchical_results.json
  Added: results/hierarchical_epoch_log.csv
  Added: results/best_model.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Downloaded results.zip
